<a href="https://colab.research.google.com/github/disCrybee/Machine-Learning/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9F%D0%B5%D1%80%D0%B5%D1%85%D1%80%D0%B5%D1%81%D1%82%D0%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**Мета:** завантажити MovieLens 100K, побудувати SVD і KNNBasic, підібрати гіперпараметри, порівняти за RMSE/MAE, сформувати топ-10 рекомендацій.

**Дані:** вбудований `ml-100k` з бібліотеки Surprise.

## 1. Підготовка середовища

In [ ]:
%pip install -q numpy scipy scikit-learn pandas scikit-surprise

In [ ]:
import sys
import numpy as np
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('Surprise:', surprise.__version__)

## 2. Завантаження та первинний аналіз

In [ ]:
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

In [ ]:
print('Унікальних користувачів:', ratings_df['user_id'].nunique())
print('Унікальних фільмів:', ratings_df['item_id'].nunique())
print('Мін/Макс/Середнє rating:', ratings_df['rating'].min(), ratings_df['rating'].max(), round(ratings_df['rating'].mean(), 3))
print('Пропуски:', ratings_df.isnull().sum().sum())
print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

**Спостереження:** ~100 тис. оцінок, сотні користувачів і тисячі фільмів. Оцінки від 1 до 5, середнє близько 3.5. Пропусків немає.

## 3. Розбиття 80/20

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('Train:', train_df.shape, 'Test:', test_df.shape)

In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('Розмір testset:', len(testset))

## 4. Гіперпараметри

In [ ]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування:**  
- `n_factors` — розмірність латентного простору.  
- `lr_all` / `reg_all` — швидкість навчання та регуляризація SVD.  
- `k` / `min_k` — кількість сусідів.  
- `sim_options` — метрика схожості та user-/item-based підхід.

## 5. GridSearchCV

In [ ]:
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('SVD best RMSE:', grid_search_svd.best_score['rmse'])
print('SVD best params:', grid_search_svd.best_params['rmse'])

In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn,
    measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('KNN best RMSE:', grid_search_knn.best_score['rmse'])
print('KNN best params:', grid_search_knn.best_params['rmse'])

In [ ]:
print('=== Результати CV ===')
print('SVD  → RMSE:', round(grid_search_svd.best_score['rmse'], 4),
      'MAE:', round(grid_search_svd.best_score['mae'], 4))
print('KNN  → RMSE:', round(grid_search_knn.best_score['rmse'], 4),
      'MAE:', round(grid_search_knn.best_score['mae'], 4))

## 6. Навчання остаточних моделей

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print('Моделі навчено.')

## 7. Оцінювання на тестовій вибірці

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD  Test RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNN  Test RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

In [ ]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')
print(results)

**Порівняння з CV:** різниця між CV і тестом зазвичай невелика. Модель обираємо за CV RMSE.

## 8. Топ-10 рекомендацій

In [ ]:
user_id = '196'

# Обираємо кращу за CV RMSE
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    print('Обрано SVD')
else:
    best_model = best_knn
    print('Обрано KNNBasic')

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f'Користувач {user_id} відсутній у trainset. Оберіть іншого.')
else:
    # Фільми, які користувач уже оцінив
    rated_items = set([trainset.to_raw_iid(iid) for (iid, _) in trainset.ur[inner_uid]])
    all_items = set(trainset.all_items())
    all_raw_items = [trainset.to_raw_iid(iid) for iid in all_items]
    candidates = [iid for iid in all_raw_items if iid not in rated_items]

    predictions = []
    for iid in candidates:
        pred = best_model.predict(uid=user_id, iid=iid)
        predictions.append((iid, pred.est))

    top10 = sorted(predictions, key=lambda x: x[1], reverse=True)[:10]
    top10_df = pd.DataFrame(top10, columns=['item_id', 'predicted_rating'])
    print(top10_df)

## 9–10. Перевірка та висновки

- [x] Дані завантажено
- [x] Розбиття 80/20 з random_state=42
- [x] Reader, train_data, testset створені
- [x] GridSearchCV для обох моделей на train_data
- [x] Моделі навчені на повній навчальній частині
- [x] RMSE/MAE на незалежному тесті
- [x] Топ-10 рекомендацій без уже оцінених фільмів

**Висновки:**
1. Модель обирається за найкращим CV RMSE (зазвичай SVD).
2. Найкращі гіперпараметри виводяться з GridSearch.
3. Тестові RMSE/MAE близькі до CV.
4. Рекомендації не містять уже оцінених фільмів.
5. Обмеження: холодний старт (нові користувачі/фільми), залежність від розрідженості матриці.

**Додаткове завдання.** Власний рекомендаційний алгоритм можна реалізувати на основі user-based collaborative filtering з косинусною схожістю вручну (матриця користувач–фільм + топ-k сусідів).